# ENSANUT HbA1c — modeling, SHAP, one shared UMAP graph, and Leiden

This notebook keeps only scientific parameters visible. The extended implementation is stored in `src/ensanut_hba1c/`.

Arquitectura obligatoria:

```text
Complete SHAP matrix
        ↓
      PCA50
   ┌────┴───────────────────────────────────────┐
PCA1D/PCA2D                  ONE internal UMAP fuzzy-kNN graph_
                                           │
                         ┌─────────────────┼──────────────────┐
                       Leiden            UMAP2D            PHATE2D
                    (direct graph)   (visualization)   (visualization)
```

Leiden, UMAP2D, and PHATE2D preserve the same participant order and receive exactly the same adjacency matrix. **Leiden does not use UMAP coordinates**, and no second kNN graph is calculated.

## 1. Project initialization


In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import importlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
warnings.filterwarnings("ignore", category=FutureWarning)

# Detect the project root even when Jupyter starts inside notebooks/.
_current = Path.cwd().expanduser().resolve()
PROJECT_ROOT = next(
    (
        candidate for candidate in [_current, *_current.parents]
        if (candidate / "src" / "ensanut_hba1c").is_dir()
        and (candidate / "data" / "input").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project folder or from notebooks/.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Force the local project code even if the kernel previously loaded
# another version of ensanut_hba1c.
for _module_name in list(sys.modules):
    if _module_name == "ensanut_hba1c" or _module_name.startswith("ensanut_hba1c."):
        sys.modules.pop(_module_name, None)
importlib.invalidate_caches()

from ensanut_hba1c.paths import ProjectPaths
PATHS = ProjectPaths(PROJECT_ROOT).ensure()
OUTPUT_DIR = PATHS.notebook1_results
HANDOFF_DIR = PATHS.handoff_dir

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_DIR:", PATHS.input_dir)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Input files and global parameters


In [ ]:
# Place only these two files inside data/input/.
DATA_FILENAME = "ENSANUT_2024_mx.csv"
DICTIONARY_FILENAME = "ENSANUT_2024.csv"

TARGET_COLUMN = "HB1AC"
WEIGHT_COLUMN = None
ID_COLUMNS = ["FOLIO_I", "FOLIO_INT"]
RANDOM_STATE = 42
N_SPLITS = 5

RUN_XGBOOST_CROSS_VALIDATION = True
RUN_MELLON_DENSITY = False
RUN_CLINICAL_REPORT = True
RUN_METABOLIC_REPORTS = True

from ensanut_hba1c.io import load_ensanut_inputs
ensanut_df, data_dictionary_df = load_ensanut_inputs(
    PATHS.input_dir,
    data_filename=DATA_FILENAME,
    dictionary_filename=DICTIONARY_FILENAME,
)
print(f"ENSANUT matrix: {ensanut_df.shape[0]:,} × {ensanut_df.shape[1]:,}")
print(f"Dictionary: {data_dictionary_df.shape[0]:,} × {data_dictionary_df.shape[1]:,}")
display(ensanut_df.head())


## 3. HbA1c regression matrix


In [ ]:
from ensanut_hba1c.modeling import prepare_model_matrix

REDUNDANT_AGE_COLUMNS = [
    "gpoedad", "annac", "mesnac", "fecha_nacimiento", "H0304A", "H0304M",
    "FECH_NAC", "edad", "AEDAD", "A0302", "MESES",
]
TARGET_DERIVED_COLUMNS = [
    "hba1c_class_cat", "hba1c_class_bin", "clasif_hba1c_cat", "clasif_hba1c_bin",
    "preferred_combined_class_bin", "preferred_combined_class_cat",
    "clasif_combinada_pref_bin", "clasif_combinada_pref_cat",
    "HbA1c_Status", "Estado_HbA1c",
]
REGRESSION_EXCLUDE_COLUMNS = sorted(set(
    REDUNDANT_AGE_COLUMNS + TARGET_DERIVED_COLUMNS + ["_merge"]
))
MAX_CATEGORICAL_LEVELS = 100

X, y, sample_weights, model_audit, hba1c_cohort_df = prepare_model_matrix(
    ensanut_df,
    target_col=TARGET_COLUMN,
    weight_col=WEIGHT_COLUMN,
    id_columns=ID_COLUMNS,
    exclude_columns=REGRESSION_EXCLUDE_COLUMNS,
    max_categorical_levels=MAX_CATEGORICAL_LEVELS,
)
print("HbA1c cohort:", hba1c_cohort_df.shape)
print("Model matrix:", X.shape)
display(pd.DataFrame([model_audit]))


## 4. XGBoost and OOF evaluation


In [ ]:
from ensanut_hba1c.modeling import cross_validate_xgb, plot_oof_predictions

XGB_DEVICE = "cpu"
XGB_N_JOBS = 4
XGB_PARAMS = {
    "n_estimators": 1339,
    "learning_rate": 0.01109231653987982,
    "max_depth": 15,
    "min_child_weight": 10,
    "subsample": 0.604285125906042,
    "colsample_bytree": 0.9741262828257075,
    "gamma": 2.0067107351969584,
    "reg_alpha": 0.0032372287050650432,
    "reg_lambda": 0.00943910513138246,
}

if RUN_XGBOOST_CROSS_VALIDATION:
    oof_prediction, fold_metrics, global_oof_metrics = cross_validate_xgb(
        X, y, XGB_PARAMS, sample_weights, N_SPLITS, RANDOM_STATE,
        device=XGB_DEVICE, n_jobs=XGB_N_JOBS,
    )
    fold_metrics.to_csv(OUTPUT_DIR / "xgboost_oof_fold_metrics.csv", index=False)
    global_oof_metrics.to_csv(OUTPUT_DIR / "xgboost_oof_global_metrics.csv", index=False)
    pd.DataFrame({
        "observed_HB1AC": y,
        "predicted_HB1AC_OOF": oof_prediction,
    }).to_csv(OUTPUT_DIR / "xgboost_oof_predictions.csv", index=False)
    oof_figure = plot_oof_predictions(
        y, oof_prediction, OUTPUT_DIR / "xgboost_oof_observed_vs_predicted.png"
    )
    display(fold_metrics)
    display(global_oof_metrics)
    display(oof_figure)
    plt.close(oof_figure)
else:
    oof_prediction = np.full(len(y), np.nan)


## 5. Final model and complete SHAP matrix


In [ ]:
from ensanut_hba1c.modeling import fit_final_xgb_and_shap, select_original_shap_matrix

SHAP_MAX_ROWS = None              # None = all participants.
SHAP_MAX_DISPLAY = 25
NORMALIZED_BEESWARM_TOP_N = 25    # Visualization only.
TOP_SHAP_FEATURES = None          # None = ALL SHAP variables for clustering.

shap_results = fit_final_xgb_and_shap(
    X, y, hba1c_cohort_df, XGB_PARAMS, sample_weights,
    output_dir=OUTPUT_DIR / "shap",
    max_rows=SHAP_MAX_ROWS,
    random_state=RANDOM_STATE,
    device=XGB_DEVICE,
    n_jobs=XGB_N_JOBS,
    max_display=SHAP_MAX_DISPLAY,
    normalized_top_n=NORMALIZED_BEESWARM_TOP_N,
)

X_shap = shap_results["X_shap"]
cohort_shap = shap_results["cohort_shap"].reset_index(drop=True)
shap_source_rows = shap_results["shap_source_rows"]
original_shap_matrix, shap_feature_names, shap_feature_indices, shap_clustering_audit = (
    select_original_shap_matrix(shap_results["shap_values"], top_n=TOP_SHAP_FEATURES)
)
if RUN_XGBOOST_CROSS_VALIDATION:
    cohort_shap["HbA1c_OOF_prediction"] = oof_prediction[shap_source_rows]

if TOP_SHAP_FEATURES is None and original_shap_matrix.shape[1] != X_shap.shape[1]:
    raise AssertionError("Not all SHAP variables were retained.")

shap_clustering_audit.to_csv(
    OUTPUT_DIR / "shap_features_used_for_clustering.csv", index=False
)
np.save(OUTPUT_DIR / "shap_values_all_features.npy", original_shap_matrix)
print("SHAP participants:", original_shap_matrix.shape[0])
print("SHAP variables used:", original_shap_matrix.shape[1])
display(shap_clustering_audit.head(25))


## 6. Editable parameters: PCA50

In [ ]:
from ensanut_hba1c.config import (
    LeidenConfig,
    PCAConfig,
    PHATEConfig,
    ReductionPipelineConfig,
    UMAPEmbeddingConfig,
    UMAPGraphConfig,
)

PCA_CONFIG = PCAConfig(
    n_components=50,                   # Shared input for all branches.
    whiten=False,
    svd_solver="randomized",          # auto | full | randomized | arpack
    tol=0.0,
    iterated_power="auto",
    n_oversamples=10,
    power_iteration_normalizer="auto",# auto | QR | LU | none
    copy=True,
    random_state=RANDOM_STATE,
)

## 7. Editable parameters: single UMAP fuzzy-kNN `graph_` built from PCA50

In [ ]:
# THESE PARAMETERS BUILD THE PROJECT'S ONLY GRAPH.
# The same graph_ is reused by Leiden, UMAP2D, and PHATE2D.
# For Leiden, n_neighbors and metric in this block define the clustering graph.
SHARED_UMAP_GRAPH_CONFIG = UMAPGraphConfig(
    n_neighbors=50,                    # k of the shared graph.
    metric="euclidean",               # euclidean | cosine | manhattan | correlation ...
    metric_kwds=None,
    local_connectivity=10.0,
    set_op_mix_ratio=1.0,              # 1.0 fuzzy union; 0.0 fuzzy intersection.
    angular_rp_forest=False,
    force_approximation_algorithm=True,
    unique=False,
    low_memory=True,
    disconnection_distance=None,
    random_state=RANDOM_STATE,
    transform_seed=RANDOM_STATE,
    n_jobs=1,
    verbose=False,
)

## 8. Editable parameters: visual UMAP2D optimized on the same `graph_`

In [ ]:
# These parameters ONLY modify the UMAP2D visualization.
# They do not recalculate neighbors, modify graph_, or affect Leiden.
UMAP2D_CONFIG = UMAPEmbeddingConfig(
    n_components=2,
    min_dist=1.0,
    spread=1.0,
    learning_rate=1.0,
    init="spectral",                  # spectral | random
    n_epochs=None,
    repulsion_strength=1.0,
    negative_sample_rate=5,
    output_metric="euclidean",
    output_metric_kwds=None,
    random_state=RANDOM_STATE,
    verbose=False,
)

## 9. Editable parameters: Leiden applied directly to the same internal `graph_`

In [ ]:
# There is no KNN_CONFIG and no clustering UMAP representation.
# Leiden directly uses the edges and fuzzy weights from UMAP's internal graph_.
LEIDEN_CONFIG = LeidenConfig(
    partition_type="RBConfiguration", # RBConfiguration | CPM | Modularity
    resolution=0.7,#0.2,
    n_iterations=-1,                  # -1 = hasta convergencia
    max_comm_size=0,                  # 0 = no limit
    seed=RANDOM_STATE,
)

## 10. Editable parameters: PHATE2D applied to the same internal `graph_`

In [ ]:
# PHATE receives exactly the same UMAP fuzzy graph_ used by Leiden and UMAP2D.
# knn_dist must remain "precomputed_affinity": no additional kNN is calculated.
PHATE_CONFIG = PHATEConfig(
    # Output
    n_components=2,

    # Native PHATE parameters exposed for control and auditing.
    # The neighborhood is already fixed by SHARED_UMAP_GRAPH_CONFIG.n_neighbors.
    knn=SHARED_UMAP_GRAPH_CONFIG.n_neighbors,
    decay=40,                        # None desactiva el kernel alpha-decay
    n_landmark=2000,                 # None desactiva landmarks

    # Diffusion
    t=10,                       # "auto" o entero positivo
    gamma=1.0,                       # -1 a 1; 1=log potential, 0=sqrt potential
    n_pca=None,                      # None: no internal PCA; PCA50 is already used

    # Afinidad precomputada y MDS
    mds_solver="sgd",               # sgd | smacof
    knn_dist="precomputed_affinity",# obligatorio para reutilizar el graph_
    knn_max=None,                    # optional PHATE neighbor limit
    mds_dist="euclidean",           # euclidean | cosine | another SciPy metric
    mds="metric",                   # classic | metric | nonmetric

    # Execution and landmarks
    n_jobs=1,                        # -1 uses all cores
    random_state=RANDOM_STATE,
    random_landmarking=False,        # False=spectral; True=muestreo aleatorio
    verbose=1,

    # Automatic t selection
    t_max=100,                       # maximum t evaluated if t is later changed to "auto"
    plot_optimal_t=False,            # True displays Von Neumann entropy

    # Advanced parameters passed by PHATE to graphtools.Graph
    graph_kwargs={
        "thresh": 1e-4,
    },
)

REDUCTION_CONFIG = ReductionPipelineConfig(
    pca=PCA_CONFIG,
    shared_umap_graph=SHARED_UMAP_GRAPH_CONFIG,
    umap2d=UMAP2D_CONFIG,
    leiden=LEIDEN_CONFIG,
    phate=PHATE_CONFIG,
)

## 11. Build the graph once, run Leiden directly, and generate the 2D views

In [ ]:
from ensanut_hba1c.reduction_pipeline import run_reduction_and_clustering

METHOD_DIR = OUTPUT_DIR / "dimensionality_and_clustering"
reduction_results = run_reduction_and_clustering(
    original_shap_matrix,
    REDUCTION_CONFIG,
    output_dir=METHOD_DIR,
)

representation_df = reduction_results["table"].reset_index(drop=True)
cluster_df = pd.concat(
    [cohort_shap.reset_index(drop=True), representation_df],
    axis=1,
)

shared_hash = reduction_results["shared_umap_graph"]["graph_fingerprint"]
print("Effective PCA components:", reduction_results["pca"]["effective_components"])
print("graph_ compartido:", reduction_results["shared_umap_graph"]["graph"].shape)
print("Neighbors in the shared graph_:", reduction_results["shared_umap_graph"]["effective_neighbors"])
print("graph_ SHA-256 fingerprint:", shared_hash)
print("Visual UMAP2D from the same graph_:", reduction_results["umap2d"]["coordinates"].shape)
print("Visual PHATE2D from the same graph_:", reduction_results["phate"]["phate2d"].shape)
print("Edges used directly by Leiden:", reduction_results["leiden_graph"]["n_edges"])
print("Leiden modularity:", reduction_results["leiden"]["modularity"])
print("Microclusters:", reduction_results["leiden"]["n_clusters"])

assert reduction_results["umap2d"]["graph_fingerprint"] == shared_hash
assert reduction_results["phate"]["graph_fingerprint"] == shared_hash
assert reduction_results["leiden_graph"]["graph_fingerprint"] == shared_hash
print("VERIFIED: direct Leiden, UMAP2D, and PHATE2D used exactly the same graph_.")

display(
    cluster_df["Cluster_SHAP"].value_counts().sort_index().rename("n").to_frame()
)

# EARLY HANDOFF: generated as soon as the microclusters exist.
# This allows Notebook 2 to run even if an optional clinical section fails later.
from ensanut_hba1c.io import export_notebook_handoff

cluster_df.to_csv(OUTPUT_DIR / "complete_cluster_dataset.csv", index=False)
early_handoff_paths = export_notebook_handoff(
    cluster_df,
    data_dictionary_df,
    HANDOFF_DIR,
    cluster_column="Cluster_SHAP",
    source_notebook="01_modeling_clustering.ipynb (automatic after clustering)",
)
print("Automatic handoff ready for Notebook 2:", HANDOFF_DIR)


## 12. Compare PCA1D, PCA2D, UMAP2D, and PHATE2D


In [ ]:
from ensanut_hba1c.reduction_visualization import (
    plot_reduction_comparison, save_figure_bundle,
)

COMPARISON_FIGURE_SIZE = (12.5, 9.5)
COMPARISON_POINT_SIZE = 7.0
COMPARISON_ALPHA = 0.75
PCA1D_DISPLAY_JITTER = 0.06
PUBLICATION_DPI = 600

comparison_figure = plot_reduction_comparison(
    representation_df,
    cluster_col="Cluster_SHAP",
    figure_size=COMPARISON_FIGURE_SIZE,
    point_size=COMPARISON_POINT_SIZE,
    alpha=COMPARISON_ALPHA,
    pca1d_jitter=PCA1D_DISPLAY_JITTER,
    random_state=RANDOM_STATE,
)
save_figure_bundle(
    comparison_figure,
    OUTPUT_DIR / "comparison_pca1d_pca2d_umap2d_phate2d",
    dpi=PUBLICATION_DPI,
)
display(comparison_figure)
plt.close(comparison_figure)


## 13. Glycemic classification and metabolic syndrome


In [ ]:
from ensanut_hba1c.clinical_visualization import (
    derive_glycemic_status, derive_metabolic_syndrome,
)

HBA1C_COLUMN = "HB1AC"
GLUCOSE_COLUMN = "GLU_SUERO"
WAIST_COLUMN = "Cintura_cm"
TRIGLYCERIDES_COLUMN = "TRIG"
HDL_COLUMN = "COL_HDL"
SYSTOLIC_BP_COLUMN = "PAS_mmHg"
DIASTOLIC_BP_COLUMN = "PAD_mmHg"
SEX_COLUMN = "H0302"
MALE_VALUE = 1
FEMALE_VALUE = 2

GLYCEMIC_NORMAL_UPPER = 5.7
GLYCEMIC_DIABETES_LOWER = 6.5
WAIST_MALE_CUTOFF = 90.0
WAIST_FEMALE_CUTOFF = 80.0
TRIGLYCERIDES_CUTOFF = 150.0
HDL_MALE_CUTOFF = 40.0
HDL_FEMALE_CUTOFF = 50.0
SYSTOLIC_BP_CUTOFF = 130.0
DIASTOLIC_BP_CUTOFF = 85.0
FASTING_GLUCOSE_CUTOFF = 100.0
METABOLIC_SYNDROME_MIN_CRITERIA = 3

cluster_df = derive_glycemic_status(
    cluster_df,
    hba1c_col=HBA1C_COLUMN,
    normal_upper=GLYCEMIC_NORMAL_UPPER,
    diabetes_lower=GLYCEMIC_DIABETES_LOWER,
)

required_metabolic = [
    SEX_COLUMN, WAIST_COLUMN, TRIGLYCERIDES_COLUMN, HDL_COLUMN,
    SYSTOLIC_BP_COLUMN, DIASTOLIC_BP_COLUMN, GLUCOSE_COLUMN,
]
missing_metabolic = [column for column in required_metabolic if column not in cluster_df]
if missing_metabolic:
    print("Metabolic syndrome skipped; missing:", missing_metabolic)
else:
    cluster_df = derive_metabolic_syndrome(
        cluster_df,
        sex_col=SEX_COLUMN, male_value=MALE_VALUE, female_value=FEMALE_VALUE,
        waist_col=WAIST_COLUMN, triglycerides_col=TRIGLYCERIDES_COLUMN,
        hdl_col=HDL_COLUMN, systolic_bp_col=SYSTOLIC_BP_COLUMN,
        diastolic_bp_col=DIASTOLIC_BP_COLUMN, glucose_col=GLUCOSE_COLUMN,
        waist_male_cutoff=WAIST_MALE_CUTOFF,
        waist_female_cutoff=WAIST_FEMALE_CUTOFF,
        triglycerides_cutoff=TRIGLYCERIDES_CUTOFF,
        hdl_male_cutoff=HDL_MALE_CUTOFF,
        hdl_female_cutoff=HDL_FEMALE_CUTOFF,
        systolic_bp_cutoff=SYSTOLIC_BP_CUTOFF,
        diastolic_bp_cutoff=DIASTOLIC_BP_CUTOFF,
        glucose_cutoff=FASTING_GLUCOSE_CUTOFF,
        minimum_criteria=METABOLIC_SYNDROME_MIN_CRITERIA,
    )
    display(cluster_df["Metabolic_Syndrome_Status"].value_counts(dropna=False))


## 14. Publication PHATE2D figure


In [ ]:
from ensanut_hba1c.reports.nature_four_panel import plot_umapgraph_phate_four_panels_nature

GLYCEMIC_COLORS = {
    "Normal": "#2E7D32", "Prediabetes": "#F9A825", "Diabetes": "#C62828",
}
METABOLIC_STATUS_COLORS = {
    "No metabolic syndrome": "#2878B5",
    "Metabolic syndrome": "#D73027",
    "Not classifiable": "#BDBDBD",
}
FOUR_PANEL_FIGSIZE = (8.8, 6.4)
FOUR_PANEL_POINT_SIZE = 3.6
FOUR_PANEL_ALPHA = 0.86

if "Metabolic_Syndrome_Status" in cluster_df:
    publication_figure = plot_umapgraph_phate_four_panels_nature(
        cluster_df,
        cluster_col="Cluster_SHAP",
        hba1c_col=TARGET_COLUMN,
        glycemic_col="HbA1c_Status",
        metabolic_col="Metabolic_Syndrome_Status",
        glycemic_colors=GLYCEMIC_COLORS,
        metabolic_colors=METABOLIC_STATUS_COLORS,
        figure_size=FOUR_PANEL_FIGSIZE,
        point_size=FOUR_PANEL_POINT_SIZE,
        alpha=FOUR_PANEL_ALPHA,
    )
    save_figure_bundle(
        publication_figure,
        OUTPUT_DIR / "phate2d_from_umap_graph_four_panel",
        dpi=1200,
    )
    display(publication_figure)
    plt.close(publication_figure)


## 15. Single manual macrocluster definition


In [ ]:
# EDIT ONLY THIS DICTIONARY after reviewing the microclusters.
SHAP_CLUSTER_MERGE_MAP = {
    # 0: 0,
    # 1: 0,
    # 2: 1,
}

current_microclusters = sorted(
    int(value) for value in cluster_df["Cluster_SHAP"].dropna().unique()
)
display(pd.DataFrame({
    "microcluster": current_microclusters,
    "macrocluster_to_assign": pd.Series([pd.NA] * len(current_microclusters), dtype="Int64"),
}))

if SHAP_CLUSTER_MERGE_MAP:
    normalized_map = {int(k): int(v) for k, v in SHAP_CLUSTER_MERGE_MAP.items()}
    missing = sorted(set(current_microclusters) - set(normalized_map))
    extra = sorted(set(normalized_map) - set(current_microclusters))
    if missing or extra:
        raise ValueError(f"Incomplete map. Missing={missing}; unknown={extra}")
    cluster_df["Cluster_SHAP_macro"] = cluster_df["Cluster_SHAP"].map(normalized_map).astype("Int64")
    pd.DataFrame(sorted(normalized_map.items()), columns=[
        "Cluster_SHAP", "Cluster_SHAP_macro"
    ]).to_csv(OUTPUT_DIR / "manual_micro_to_macro_map.csv", index=False)
else:
    cluster_df["Cluster_SHAP_macro"] = pd.Series(pd.NA, index=cluster_df.index, dtype="Int64")
    print("Macroclusters are not defined yet. Notebook 2 can run without them.")


## 16. Optional clinical and metabolic reports


In [ ]:
if RUN_CLINICAL_REPORT:
    from ensanut_hba1c.reports.clinical import generate_clinical_cluster_report
    clinical_candidates = [
        "HB1AC", "GLU_SUERO", "ALBU", "IMC", "INSULINA", "COL_HDL", "COL_LDL",
        "TRIG", "AC_URICO", "Peso_kg", "Talla_cm", "Cintura_cm", "PAS_mmHg",
        "PAD_mmHg", "CREAT", "H0303",
    ]
    clinical_variables = [column for column in clinical_candidates if column in cluster_df]

    # Complete these maps when publication labels and units are required.
    CLINICAL_LABEL_MAP = {}
    CLINICAL_UNIT_MAP = {
        "HB1AC": "%", "GLU_SUERO": "mg/dL", "IMC": "kg/m²",
        "COL_HDL": "mg/dL", "COL_LDL": "mg/dL", "TRIG": "mg/dL",
        "Peso_kg": "kg", "Talla_cm": "cm", "Cintura_cm": "cm",
        "PAS_mmHg": "mmHg", "PAD_mmHg": "mmHg",
    }
    CLINICAL_SEX_COLUMN = SEX_COLUMN if SEX_COLUMN in cluster_df else None

    if clinical_variables:
        clinical_report = generate_clinical_cluster_report(
            cluster_df,
            variables=clinical_variables,
            cluster_col="Cluster_SHAP",
            label_map=CLINICAL_LABEL_MAP,
            unit_map=CLINICAL_UNIT_MAP,
            sex_col=CLINICAL_SEX_COLUMN,
            female_value=FEMALE_VALUE,
            output_dir=OUTPUT_DIR / "clinical_microcluster_report",
            heatmap_vmin=-2.5,
            heatmap_vmax=2.5,
        )
        display(clinical_report["table"])
        display(clinical_report["figure"])
        plt.close(clinical_report["figure"])

if RUN_METABOLIC_REPORTS and "Metabolic_Syndrome_Status" in cluster_df:
    from ensanut_hba1c.reports.metabolic import generate_metabolic_cluster_report
    metabolic_report = generate_metabolic_cluster_report(
        df=cluster_df,
        cluster_col="Cluster_SHAP",
        output_dir=OUTPUT_DIR / "metabolic_microcluster_report",
        file_prefix="leiden_microclusters",
        cluster_label="Leiden microcluster",
    )
    display(metabolic_report["summary"])
    for key in ("average_figure", "stacked_bar_figure", "heatmap_figure"):
        display(metabolic_report[key])
        plt.close(metabolic_report[key])

## 17. Handoff to Notebook 2 and results ZIP


In [ ]:
from ensanut_hba1c.io import export_notebook_handoff

cluster_df.to_csv(OUTPUT_DIR / "complete_cluster_dataset.csv", index=False)
handoff_paths = export_notebook_handoff(
    cluster_df,
    data_dictionary_df,
    HANDOFF_DIR,
    cluster_column="Cluster_SHAP",
    source_notebook="01_modeling_clustering.ipynb (final refresh)",
)
for name, path in handoff_paths.items():
    print(name, "->", path)

RESULTS_ZIP = shutil.make_archive(
    base_name=str(PROJECT_ROOT / "ENSANUT_HbA1c_notebook1_results"),
    format="zip",
    root_dir=OUTPUT_DIR,
)
print("ZIP:", RESULTS_ZIP)
print("Next notebook:", PROJECT_ROOT / "notebooks" / "02_naive_bayes_followup.ipynb")


In [ ]:
# PRE-RENDERED FIGURE GALLERY
# No saved image outputs were present in the uploaded notebook.


## Pre-rendered figure gallery

No saved image outputs were present in the uploaded notebook, so there are no figures that can be embedded without executing the analysis.